# Inflation-guard pipeline, live check
`inflation_guard_pipeline.py` (repo root, moved there on 2026-11-15) (pre-registered rule in `results/preregistered_guard.json`). Live (GPU) on 12 official photos that cover every stage (bar, agreement, escalation, skip).
Checks: (1) the guard is silent on clean data (factor 1.0), so stage, place and query count equal the replay of AgreementQueryPipeline on the stored dump; (2) the far-candidate level computed live from the queries equals the level computed offline from the signal table (same 6 standard queries).

In [1]:
import os, sys, time
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "tests/Test_Iteration_7/changed_src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from train.checks import replay
import train.checks.scale_robust as sr
T7 = ROOT / "tests/Test_Iteration_7/results"
replayed = {pid: replay.decide_current(ph) for pid, ph in replay.load_set(T7 / "rich_dump/official150").items()}
by_stage = {}
for pid, r in replayed.items():
    by_stage.setdefault(r["stage"], []).append(pid)
photo_ids = by_stage["bar"][:4] + by_stage["agreement"][:3] + by_stage["escalation"][:3] + by_stage["skip"][:2]
print({k: len(v) for k, v in by_stage.items()}, "->", photo_ids)
raw = pd.read_csv(T7 / "signals_ours_official150.csv.gz")
keys = {(round(h, 2), round(s, 2)) for h, s in replay.STANDARD}
std = raw[[(round(h, 2), round(s, 2)) in keys for h, s in zip(raw.heading, raw.scale)]]
offline_level = sr.far_levels(std).set_index("photo").far_inl

{'bar': 126, 'agreement': 7, 'skip': 16, 'escalation': 1} -> [1, 2, 5, 6, 4, 36, 72, 148, 7, 11]


In [2]:
from inflation_guard_pipeline import InflationGuardPipeline
pipeline = InflationGuardPipeline("official", "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                  matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                                  map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None)
pipeline.setup(500, 7)
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 113 s


data: official | 379 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 6 (headings [-5.0, 0.0, 5.0], scale factors [1.0, 1.05]); K and M are per query; 1 queries matched together
decision: answer at >= 8 inliers or >= 2 queries agreeing within 1.0 sigma (each >= 3 inliers)
escalation: on; 27 wide queries, answer at >= 11 inliers
inflation guard: prior 0.169 (weight 30 photos), dead zone x1.5; bars scale with the running far-candidate level


In [3]:
rows = []
for pid in photo_ids:
    t0 = time.time()
    r = pipeline.run(test_set.items[pid], test_set.priors[pid])
    err = float(np.hypot(r["x"] - test_set.truths[pid][0], r["y"] - test_set.truths[pid][1]) / test_set.sigmas[pid])
    o = replayed[pid]
    rows.append(dict(photo=pid, live_stage=r["stage"], replay_stage=o["stage"], live_queries=r["n_queries_run"], replay_queries=o["n_queries"],
                     live_err_sigma=round(err, 2), replay_err_sigma=round(float(o["err_sigma"]), 2), factor=r["guard_factor"],
                     live_level=round(r["far_level"], 3), offline_level=round(float(offline_level.loc[pid]), 3), seconds=round(time.time() - t0, 1)))
live = pd.DataFrame(rows)
live["same_stage"] = live.live_stage == live.replay_stage
live["same_place"] = (live.live_err_sigma - live.replay_err_sigma).abs() < 0.2
live["same_level"] = (live.live_level - live.offline_level).abs() < 0.02
live

,photo,live_stage,replay_stage,live_queries,replay_queries,live_err_sigma,replay_err_sigma,factor,live_level,offline_level,seconds,same_stage,same_place,same_level
0,1,bar,bar,6,6,0.60,0.60,1.0,0.833,0.833,4.9,True,True,True
1,2,bar,bar,6,6,0.76,0.76,1.0,0.000,0.000,3.2,True,True,True
2,5,bar,bar,6,6,0.83,0.83,1.0,0.037,0.037,3.2,True,True,True
3,6,bar,bar,6,6,0.45,0.45,1.0,0.071,0.071,5.6,True,True,True
4,4,agreement,agreement,6,6,0.64,0.64,1.0,0.000,0.000,5.6,True,True,True
5,36,agreement,agreement,6,6,0.42,0.42,1.0,0.000,0.000,5.7,True,True,True
6,72,agreement,agreement,6,6,0.73,0.73,1.0,0.000,0.000,5.6,True,True,True
7,148,escalation,escalation,27,27,0.65,0.65,1.0,0.000,0.000,18.7,True,True,True
8,7,skip,skip,27,27,1.00,1.00,1.0,0.000,0.000,13.4,True,True,True
9,11,skip,skip,27,27,11.98,11.98,1.0,0.000,0.000,24.6,True,True,True


In [4]:
print("same stage:", int(live.same_stage.sum()), "of", len(live), "| same place:", int(live.same_place.sum()), "| same level:", int(live.same_level.sum()),
      "| factors:", sorted(set(live.factor)))
live.to_csv(T7 / "inflation_guard_live_check.csv", index=False)

same stage: 10 of 10 | same place: 10 | same level: 10 | factors: [1.0]
